# PTTEP Data Retrieval

# Task 1: Install all necessary programs and packages

# Cell 0: Install Yhoo Finance program

In [ ]:
%pip install -q --upgrade "numpy>=2.2,<2.4" pandas scipy numba yfinance ydata-profiling requests

# Restrat Runtime Session (Tab Above)



*   👉 Runtime
*   👉 Restart Session




# Task 2: Load Ready Data (SET, 1Y-YC, 10YR-YC Government Bond) to Content

# Cell 1: Retreive SET and Gov Bond Yield (Manual)

In [ ]:
# Cell 1: Brows Investing.com web

from IPython.display import HTML, display

url = "https://www.investing.com/indices/major-indices"

display(HTML(
    f'<a href="{url}" target="_blank" rel="noopener noreferrer">'
    'Open Investing.com — Major Indices'
    '</a>'
))

# Cell 2A: Read SET and Gov Bond Yield (GBY) from Github

In [ ]:
from pathlib import Path
import requests
import pandas as pd
from IPython.display import display

SAVE_DIR = Path("/content")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Kiekulkunya/KU_FINMODEL/refs/heads/main/"
)

FILES = {
    "SET_Index_Historical_Data.csv":
        BASE_URL + "SET%20Index%20Historical%20Data.csv",

    "Thailand_10Y_Bond_Yield_Historical_Data.csv":
        BASE_URL + "Thailand%2010-Year%20Bond%20Yield%20Historical%20Data.csv",

    "Thailand_1Y_Bond_Yield_Historical_Data.csv":
        BASE_URL + "Thailand%201-Year%20Bond%20Yield%20Historical%20Data.csv",
}

# Download all three files.
for filename, url in FILES.items():
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    path = SAVE_DIR / filename
    path.write_bytes(response.content)
    print(f"✅ Saved: {path}")

# Load the saved files into pandas.
SET_PATH = SAVE_DIR / "SET_Index_Historical_Data.csv"
BOND_PATH = SAVE_DIR / "Thailand_10Y_Bond_Yield_Historical_Data.csv"
BOND_1Y_PATH = SAVE_DIR / "Thailand_1Y_Bond_Yield_Historical_Data.csv"

set_df = pd.read_csv(SET_PATH)
bond_df = pd.read_csv(BOND_PATH)
bond_1y_df = pd.read_csv(BOND_1Y_PATH)

for label, df in [
    ("SET Index", set_df),
    ("Thailand 10-Year Government Bond Yield", bond_df),
    ("Thailand 1-Year Government Bond Yield", bond_1y_df),
]:
    print(f"\n{label}: {len(df):,} rows")
    display(df.head())

# skip (👉) Cell 2B: Load and Read file from content (Option)

In [ ]:
import pandas as pd
from IPython.display import display

# Read local Colab files.
set_df = pd.read_csv("/content/SET_Index_Historical_Data.csv")
bond_df = pd.read_csv("/content/Thailand_10Y_Bond_Yield_Historical_Data.csv")
bond_1y_df = pd.read_csv("/content/Thailand_1Y_Bond_Yield_Historical_Data.csv")


def prepare_series(df, value_name):
    df = df.copy()
    df.columns = df.columns.str.strip()

    required = {"Date", "Price"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"{value_name}: expected Date and Price columns; "
            f"found {df.columns.tolist()}"
        )

    # Investing.com English CSV date format.
    df["Date"] = pd.to_datetime(
        df["Date"].astype("string").str.strip(),
        format="%m/%d/%Y",
        errors="raise",
    )

    df[value_name] = pd.to_numeric(
        df["Price"].astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="raise",
    )

    if df["Date"].isna().any():
        raise ValueError(f"{value_name}: missing dates found.")

    if df["Date"].duplicated().any():
        raise ValueError(f"{value_name}: duplicate dates found.")

    return df[["Date", value_name]]


# Prepare each series.
set_series = prepare_series(set_df, "SET_Index")
bond_series = prepare_series(bond_df, "Thailand_10Y_Yield_Pct")
bond_1y_series = prepare_series(bond_1y_df, "Thailand_1Y_Yield_Pct")

# Match by date, keeping all dates.
# Missing observations remain blank; no interpolation is applied.
combined_df = (
    set_series
    .merge(
        bond_series,
        on="Date",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        bond_1y_series,
        on="Date",
        how="outer",
        validate="one_to_one",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

# Save using the filename expected by your next notebook cell.
COMBINED_PATH = "/content/SET_and_Thailand_10Y_Bond.csv"

combined_df.to_csv(
    COMBINED_PATH,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8-sig",
)

print(f"✅ Saved {len(combined_df):,} rows to {COMBINED_PATH}")
print("\nMissing observations:")
print(combined_df.isna().sum())

display(combined_df.head(10))
display(combined_df.tail(10))

# Cell 3: Combine SET ; 1GBY; 10 GBY

In [ ]:
import pandas as pd

def prepare_series(df, value_name):
    df = df.copy()
    df.columns = df.columns.str.strip()

    required = {"Date", "Price"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"Expected columns {required}; found {df.columns.tolist()}"
        )

    df["Date"] = pd.to_datetime(
        df["Date"].astype(str).str.strip(),
        format="%m/%d/%Y",
        errors="raise",
    )

    df[value_name] = pd.to_numeric(
        df["Price"].astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="raise",
    )

    if df["Date"].isna().any():
        raise ValueError(f"Missing dates found in {value_name}")

    if df["Date"].duplicated().any():
        raise ValueError(f"Duplicate dates found in {value_name}")

    return df[["Date", value_name]]


# Prepare all three series.
set_series = prepare_series(set_df, "SET_Index")
bond_series = prepare_series(bond_df, "Thailand_10Y_Yield_Pct")
bond_1y_series = prepare_series(bond_1y_df, "Thailand_1Y_Yield_Pct")

# Match by date, keeping all dates and leaving missing values blank.
combined_df = (
    set_series
    .merge(
        bond_series,
        on="Date",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        bond_1y_series,
        on="Date",
        how="outer",
        validate="one_to_one",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

# Keep the existing filename for compatibility with your next cell.
# This file now contains SET, 10-year yield, and 1-year yield.
COMBINED_PATH = "/content/SET_and_Thailand_10Y_Bond.csv"

combined_df.to_csv(
    COMBINED_PATH,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8",
)

print(f"✅ Saved {len(combined_df):,} rows to {COMBINED_PATH}")
print("\nMissing observations:")
print(combined_df.isna().sum())

display(combined_df.head(10))
display(combined_df.tail(10))

# Cell 4: Auto Load file to desktop

In [ ]:
from google.colab import files

files.download("/content/SET_and_Thailand_10Y_Bond.csv")

# Task 3:  Auto Web Scrapping Starts Here

# Cell 5 Yahoo finance auto webscapping
- PTTEP
- Exhange rate THB
- Oil
- S&P 500  


In [ ]:
import time
from pathlib import Path

import pandas as pd
import yfinance as yf

TICKERS = [
    "PTTEP.BK",       # PTT Exploration and Production
    "THBUSD=X",     # USD per THB
    "BZ=F",        # Brent crude futures
    "^GSPC",       # S&P 500
]

START = "2021-01-01"

# end is exclusive: exclude today's potentially unfinished daily prices.
END = pd.Timestamp.now(tz="Asia/Bangkok").date().isoformat()

SAVE_PATH = Path("/content/PTTEP_analysis.csv")
MAX_RETRIES = 3
RETRY_DELAY = 5
BETWEEN_TICKERS = 2

PRICE_COLS = ["Open", "High", "Low", "Close"]
VALUE_COLS = PRICE_COLS + ["Adj Close", "Volume"]
COLUMNS = ["Date", "Symbol"] + VALUE_COLS

print(f"yfinance version: {yf.__version__}")
print(f"Requested dates: {START} <= Date < {END}")


def fetch(ticker):
    last_error = ""

    for attempt in range(1, MAX_RETRIES + 1):
        # A different date request may help if Yahoo's range request fails.
        # It still uses Yahoo, so it cannot guarantee data availability.
        date_args = (
            {"start": START, "end": END}
            if attempt == 1
            else {"period": "max"}
        )
        mode = "requested dates" if attempt == 1 else "maximum history"

        print(f"\n[{ticker}] attempt {attempt}/{MAX_RETRIES}: {mode}")

        try:
            # Ticker.history returns ordinary columns for one ticker.
            df = yf.Ticker(ticker).history(
                **date_args,
                interval="1d",
                auto_adjust=False,
                actions=False,
                timeout=20,
            )

            if df is None or df.empty:
                raise ValueError("Yahoo returned no historical rows")

            df = df.copy()

            # Preserve each market's local trading date.
            dates = pd.DatetimeIndex(df.index)
            if dates.tz is not None:
                dates = dates.tz_localize(None)
            df.index = dates.normalize()
            df.index.name = "Date"

            # Apply the same date limits to both retrieval methods.
            df = df.loc[
                (df.index >= pd.Timestamp(START))
                & (df.index < pd.Timestamp(END))
            ]

            if "Close" not in df.columns:
                raise ValueError("Response has no Close column")

            # Preserve missing fields as NaN; do not manufacture prices.
            df = df.reindex(columns=VALUE_COLS)
            df[VALUE_COLS] = df[VALUE_COLS].apply(
                pd.to_numeric, errors="coerce"
            )
            df = df.dropna(subset=["Close"])

            if df.empty:
                raise ValueError("No usable Close prices in requested dates")

            df = df.loc[~df.index.duplicated(keep="last")].sort_index()
            df = df.reset_index()
            df.insert(1, "Symbol", ticker)

            print(
                f"[{ticker}] ✅ {len(df):,} rows "
                f"({df['Date'].min().date()} to "
                f"{df['Date'].max().date()})"
            )
            return df[COLUMNS], None

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            print(f"[{ticker}] {last_error}")

            if attempt < MAX_RETRIES:
                delay = RETRY_DELAY * attempt
                print(f"[{ticker}] retrying in {delay}s...")
                time.sleep(delay)

    print(f"[{ticker}] ❌ unavailable; continuing")
    return pd.DataFrame(columns=COLUMNS), last_error


frames = []
failures = {}
completed = set()
final_df = pd.DataFrame(columns=COLUMNS)

try:
    for position, ticker in enumerate(TICKERS):
        df, error = fetch(ticker)

        if not df.empty:
            frames.append(df)
            completed.add(ticker)

            # Checkpoint after every successful ticker.
            final_df = (
                pd.concat(frames, ignore_index=True)
                .drop_duplicates(["Symbol", "Date"], keep="last")
                .sort_values(["Symbol", "Date"])
                .reset_index(drop=True)
            )
            final_df.to_csv(
                SAVE_PATH,
                index=False,
                encoding="utf-8",
                date_format="%Y-%m-%d",
            )
            print(f"Checkpoint saved: {SAVE_PATH}")
        else:
            failures[ticker] = error

        if position < len(TICKERS) - 1:
            time.sleep(BETWEEN_TICKERS)

except KeyboardInterrupt:
    print("\nStopped by user. Completed downloads remain saved.")

print("\n" + "=" * 60)

if not final_df.empty:
    print(f"Saved {len(final_df):,} rows to {SAVE_PATH}")
    print("\nCoverage by symbol:")
    print(
        final_df.groupby("Symbol").agg(
            Rows=("Date", "size"),
            First_date=("Date", "min"),
            Last_date=("Date", "max"),
        ).to_string()
    )
    print("\nFirst 10 rows:")
    print(final_df.head(10).to_string(index=False))
else:
    print("No data saved by this run.")

missing = [ticker for ticker in TICKERS if ticker not in completed]

if missing:
    print("\n⚠ INCOMPLETE DATASET — missing:", ", ".join(missing))
    for ticker in missing:
        print(
            f"  {ticker}: "
            f"{failures.get(ticker, 'Interrupted or not attempted')}"
        )
else:
    print("\n✅ All requested symbols returned data.")
    print("Check the coverage dates above before using the dataset.")

# Cell 6: Combine Data PTTEP, SET, GBY (1 and 19), BZ, THBUSD, GSPC

In [ ]:
import pandas as pd
from google.colab import files

# 1. Load both saved datasets.
assets = pd.read_csv(
    "/content/PTTEP_analysis.csv",
    parse_dates=["Date"],
)
set_bond = pd.read_csv(
    "/content/SET_and_Thailand_10Y_Bond.csv",
    parse_dates=["Date"],
)

# 2. Standardize dates.
for df in (assets, set_bond):
    df["Date"] = pd.to_datetime(df["Date"], errors="raise").dt.normalize()
    if df["Date"].isna().any():
        raise ValueError("Missing dates found; check the input files.")

# 3. Check uniqueness before merging.
if assets.duplicated(["Date", "Symbol"]).any():
    raise ValueError("Duplicate Date–Symbol pairs in PTTEP_analysis.csv.")

if set_bond["Date"].duplicated().any():
    raise ValueError("Duplicate dates in SET_and_Thailand_10Y_Bond.csv.")

# 4. Convert asset data from long format to one column per symbol.
asset_prices = (
    assets.pivot(index="Date", columns="Symbol", values="Close")
    .rename(columns={
        "PTTEP.BK": "PTTEP_Close",
        "THBUSD=X": "THBUSD_Close",
        "BZ=F": "Brent_Close",
        "^GSPC": "SP500_Close",
    })
    .rename_axis(columns=None)
    .reset_index()
)

# 5. Match by date, retaining all dates from both files.
# Different market holidays will produce some missing values.
combined = (
    asset_prices.merge(
        set_bond,
        on="Date",
        how="outer",
        validate="one_to_one",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

# 6. Save the combined dataset.
OUTPUT_PATH = "/content/PTTEP_All_Series_Combined.csv"

combined.to_csv(
    OUTPUT_PATH,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8-sig",
)

print(f"✅ Saved {len(combined):,} dates to {OUTPUT_PATH}")
print("\nMissing values by column:")
print(combined.isna().sum())

display(combined.head(10))
display(combined.tail(10))

# 7. Automatically initiate download to your computer.
files.download(OUTPUT_PATH)

# Task 4: Financial Analysis Starts Here

# Cell 7: Interpolate Thai Gov Bond (1Y and 10Y)

In [ ]:
import pandas as pd
from google.colab import files
from IPython.display import display

INPUT_PATH = "/content/PTTEP_All_Series_Combined.csv"
OUTPUT_PATH = "/content/PTTEP_All_Series_Interpolated.csv"

BOND_COLS = [
    "Thailand_1Y_Yield_Pct",
    "Thailand_10Y_Yield_Pct",
]

# True reproduces your original approach of filling edge gaps.
# False leaves missing values before/after the observed yield history.
FILL_EDGES = True

# 1. Read the combined dataset.
df = pd.read_csv(INPUT_PATH)
df.columns = df.columns.str.strip()

required = ["Date", "PTTEP_Close"] + BOND_COLS
missing_cols = [col for col in required if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["Date"] = pd.to_datetime(
    df["Date"], format="%Y-%m-%d", errors="raise"
)

if df["Date"].isna().any():
    raise ValueError("Missing dates found.")

if df["Date"].duplicated().any():
    raise ValueError("Duplicate dates found.")

df = df.sort_values("Date").set_index("Date")

# 2. Validate numeric yields.
for col in BOND_COLS:
    df[col] = pd.to_numeric(
        df[col].astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="raise",
    ).astype(float)

    if df[col].notna().sum() < 2:
        raise ValueError(
            f"{col}: at least two observed yields are needed."
        )

# Identify PTTEP trading dates.
pttep_dates = df["PTTEP_Close"].notna()

if not pttep_dates.any():
    raise ValueError("No observed PTTEP closing prices found.")

before = df.loc[pttep_dates, BOND_COLS].isna().sum()

# 3. Interpolate using elapsed calendar time.
# Use all available bond observations before selecting PTTEP dates.
for col in BOND_COLS:
    originally_missing = df[col].isna()

    filled = df[col].interpolate(
        method="time",
        limit_area="inside",
    )

    if FILL_EDGES:
        filled = filled.ffill().bfill()

    df[col] = filled

    # Mark estimated values so they can be identified later.
    df[f"{col}_Imputed"] = originally_missing & filled.notna()

# 4. Keep dates with observed PTTEP prices.
# Other asset columns are retained without filling their missing values.
result = df.loc[pttep_dates].copy()

after = result[BOND_COLS].isna().sum()

print("Bond-yield missing values on PTTEP dates:")
display(pd.DataFrame({
    "Before": before,
    "After": after,
    "Filled": before - after,
}))

# 5. Save a new file, preserving the original combined CSV.
result = result.reset_index()

result.to_csv(
    OUTPUT_PATH,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8-sig",
)

print(f"✅ Saved {len(result):,} rows to {OUTPUT_PATH}")
display(result.head(10))
display(result.tail(10))

# 6. Initiate download to your browser's download folder.
files.download(OUTPUT_PATH)

# Task 5: Data Preparation and Explatory Data Analysis

# Cell 8: Basic Data Preparation by checking missing data

In [ ]:
import pandas as pd

# Load your CSV file
file_path = "/content/PTTEP_All_Series_Interpolated.csv"  # Change to your file
df = pd.read_csv(file_path)

print("="*60)
print("MISSING DATA CHECK")
print("="*60)

# Check missing data
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("\n✅ NO MISSING DATA!")
else:
    print("\n⚠️ MISSING DATA FOUND:\n")
    for col, count in missing.items():
        percentage = (count / len(df)) * 100
        print(f"{col}: {count:,} missing ({percentage:.1f}%)")

# Total summary
total_missing = df.isnull().sum().sum()
total_cells = df.size
print(f"\nTotal missing: {total_missing:,} out of {total_cells:,} cells")
print(f"Missing percentage: {(total_missing/total_cells*100):.2f}%")
print("="*60)

In [ ]:
print(df.columns.tolist())

# Cell 9: Cleaning and droping missing data by dropna all NaN

In [ ]:
import pandas as pd
from google.colab import files
from IPython.display import display

INPUT_PATH = "/content/PTTEP_All_Series_Interpolated.csv"
OUTPUT_PATH = "/content/PTTEP_All_Series_Interpolated_Clean.csv"

# Actual columns created by your previous merge cells.
SERIES = {
    "PTTEP_Close": ("PTTEP stock", "THB"),
    "Brent_Close": ("Brent crude oil", "USD"),
    "THBUSD_Close": ("THB/USD exchange rate", "USD per THB"),
    "Thailand_1Y_Yield_Pct": ("Thai 1-year bond yield", "%"),
    "Thailand_10Y_Yield_Pct": ("Thai 10-year bond yield", "%"),
    "SET_Index": ("SET Index", "points"),
    "SP500_Close": ("S&P 500", "points"),
}

# 1. Load and validate.
df = pd.read_csv(INPUT_PATH)
df.columns = df.columns.str.strip()

required = ["Date"] + list(SERIES)
missing_cols = [col for col in required if col not in df.columns]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}\n"
        f"Available columns: {df.columns.tolist()}"
    )

if df.empty:
    raise ValueError("The input file contains no rows.")

df["Date"] = pd.to_datetime(
    df["Date"], format="%Y-%m-%d", errors="raise"
)

for col in SERIES:
    df[col] = pd.to_numeric(df[col], errors="raise")

df = df.sort_values("Date").reset_index(drop=True)

if df["Date"].duplicated().any():
    raise ValueError("Duplicate dates found. Check the combined dataset.")

# 2. Show missing values.
print("=" * 60)
print("Cleaning Data: Dropping Rows with Any Missing Values")
print("=" * 60)

print(f"\nOriginal rows: {len(df):,}")
print(f"Original columns: {len(df.columns)}")

missing_before = df.isna().sum()

print("\nMissing values before cleaning:")
display(
    missing_before[missing_before > 0]
    .rename("Missing_Count")
    .to_frame()
)

# 3. Remove rows containing any missing value.
# All columns, including the imputation flags, are retained.
df_clean = df.dropna().copy().reset_index(drop=True)

if df_clean.empty:
    raise ValueError(
        "No rows remain after dropping missing values. "
        "Review the input data before saving."
    )

print(f"\nRows remaining: {len(df_clean):,}")
print(f"Rows removed: {len(df) - len(df_clean):,}")
print(f"Data retention: {len(df_clean) / len(df) * 100:.1f}%")
print(
    f"Date range: {df_clean['Date'].min().date()} "
    f"to {df_clean['Date'].max().date()}"
)
print(f"Missing values after cleaning: {df_clean.isna().sum().sum()}")

# 4. Summarize the actual data columns.
# "Latest" refers to the last retained date, not a live quote.
summary = []

for col, (label, unit) in SERIES.items():
    values = df_clean[col]

    summary.append({
        "Series": label,
        "Unit": unit,
        "Latest": values.iloc[-1],
        "Mean": values.mean(),
        "Minimum": values.min(),
        "Maximum": values.max(),
    })

print(
    f"\nSummary statistics "
    f"(latest retained date: {df_clean['Date'].iloc[-1].date()}):"
)
display(pd.DataFrame(summary).round(4))

# 5. Save and initiate download.
df_clean.to_csv(
    OUTPUT_PATH,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8-sig",
)

print(f"\n✅ Saved: {OUTPUT_PATH}")
print(f"Shape: {len(df_clean):,} rows × {len(df_clean.columns)} columns")

display(df_clean.head())

files.download(OUTPUT_PATH)

# Cell 10: Robust Data Validation

In [ ]:
# Quick preview
import pandas as pd
df_clean
print(f"\n\nTotal records: {len(df_clean):,}")
print(f"No missing values: {df_clean.isnull().sum().sum() == 0}")

In [ ]:
df_clean

# Task 6:  Drop missing data and convert FX to THB/USD

# Cell 11: drop, convert and save final file

In [ ]:
import pandas as pd
from google.colab import files

INPUT_PATH = "/content/PTTEP_All_Series_Interpolated_Clean.csv"
OUTPUT_PATH = "/content/PTTEP_All_Series_Final.csv"

# Load the latest cleaned dataset.
df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])

# Remove the old volume column if present.
df = df.drop(columns=["THBUSD=X_Volume"], errors="ignore")

# Validate the exchange-rate column.
if "THBUSD_Close" not in df.columns:
    raise ValueError("Required column 'THBUSD_Close' was not found.")

fx = pd.to_numeric(df["THBUSD_Close"], errors="raise")

if fx.isna().any() or (fx <= 0).any():
    raise ValueError("Exchange rates must be nonmissing and positive.")

# THBUSD_Close: USD per 1 THB.
# Reciprocal: THB per 1 USD.
df["FX_THB_per_USD"] = 1 / fx

# Preserve all other columns, including interpolated bond yields
# and their imputation flags.
df.to_csv(
    OUTPUT_PATH,
    index=False,
    date_format="%Y-%m-%d",
    encoding="utf-8-sig",
)

print("✅ Created FX_THB_per_USD")
print(df[["Date", "THBUSD_Close", "FX_THB_per_USD"]].head())
print(f"\n💾 Saved: {OUTPUT_PATH}")

# Initiate download to your computer.
files.download(OUTPUT_PATH)

# Check

In [ ]:
df

# Task 7: Data Cleaning and Exploratory Analysis (EDA)

# Cell 12: Render Ydata for Exploratory Data Analysis (EDA)

In [ ]:
from pathlib import Path
import pandas as pd
from ydata_profiling import ProfileReport
from google.colab import files
from IPython.display import display, HTML

INPUT_PATH = "/content/PTTEP_All_Series_Final.csv"
OUTPUT_PATH = "/content/PTTEP_All_Series_Profile_Report.html"

# Load the final dataset.
df = pd.read_csv(INPUT_PATH)
df["Date"] = pd.to_datetime(df["Date"], errors="raise")
df = df.sort_values("Date").reset_index(drop=True)

print(f"Generating report for {len(df):,} records...")

profile = ProfileReport(
    df,
    title="PTTEP Market Data — Final Dataset",
    missing_diagrams={
        "bar": True,
        "matrix": True,
        "heatmap": True,
    },
    explorative=True,
)

# Save the HTML report.
profile.to_file(OUTPUT_PATH)

file_size = Path(OUTPUT_PATH).stat().st_size / (1024 * 1024)

print(f"✅ Report saved: {OUTPUT_PATH}")
print(f"📊 Report size: {file_size:.2f} MB")

# Automatically initiate the HTML download.
files.download(OUTPUT_PATH)

# Render the report directly in the Colab output.
display(HTML("<h3>PTTEP Market Data — Profiling Report</h3>"))
profile.to_notebook_iframe()

# cell 13: Generate Boolean for Bull, Bear, and Sideway for PTTEP

1. Underlying ComponentsLet $P_t$ denote the closing price of PTTEP (PTTEP_Close) on trading day $t$:50-Day Simple Moving Average ($\text{SMA}_{50, t}$):$$\text{SMA}_{50, t} = \frac{1}{50} \sum_{i=0}^{49} P_{t-i}$$20-Day Percentage Return ($\text{Ret}_{20d, t}$):$$\text{Ret}_{20d, t} = \frac{P_t - P_{t-20}}{P_{t-20}}$$2. Binary Classification Rule (Bull vs. Bear)$$\text{PTTEP\_Classification}_t = \begin{cases} \text{Bull}, & \text{if } \left(P_t > \text{SMA}_{50, t}\right) \;\land\; \left(\text{Ret}_{20d, t} > 0.03\right) \\ \text{Bear}, & \text{if } \left(P_t \le \text{SMA}_{50, t}\right) \;\lor\; \left(\text{Ret}_{20d, t} \le 0.03\right) \end{cases}$$Where:$\land$ denotes logical AND (& in Python): both conditions must hold simultaneously for Bull.$\lor$ denotes logical OR (| in Python): if either condition fails (or during the initial warm-up rows where $\text{SMA}_{50, t}$ or $\text{Ret}_{20d, t}$ is NaN), day $t$ is classified as Bear.

In [ ]:

import pandas as pd
import numpy as np
from google.colab import files

# 1. Load dataset from Google Colab path
file_path = '/content/PTTEP_All_Series_Final.csv'
df = pd.read_csv(file_path)

# 2. Ensure chronological order by Date
df['Date_dt'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date_dt').reset_index(drop=True)

# 3. Compute technical trend and momentum indicator
sma_50 = df['PTTEP_Close'].rolling(window=50).mean()

# 4. Generate binary classification column 'PTTEP_Classification' ('Bull' vs 'Bear')
# Bull: Price >= 50-day SMA (uptrend/above average)
# Bear: Price < 50-day SMA (downtrend/below average)
df['PTTEP_Classification'] = np.where(df['PTTEP_Close'] >= sma_50, 'Bull', 'Bear')

# 5. Clean up temporary helper column and save to CSV
df = df.drop(columns=['Date_dt'])
output_path = '/content/PTTEP_all_Final_Full.csv'
df.to_csv(output_path, index=False)

# 6. Summary check (displays all columns without truncation)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Saved successfully to {output_path}")
print("\nClass Distribution:")
print(df['PTTEP_Classification'].value_counts())
print("\nPreview (All Columns):")
print(df.tail(10))

# 7. Auto-download file directly to your local computer
files.download(output_path)

# Complete Task

In [ ]:
from IPython.display import Image, display

IMAGE_URL = (
    "https://raw.githubusercontent.com/Kiekulkunya/KU_FINMODEL/main/"
    "DALL%C2%B7E%202024-11-04%2018.34.19%20-%20A%20Shiba%20Inu%20dog"
    "%20sitting%20at%20a%20desk%2C%20wearing%20a%20green%20sweater"
    "%20that%20says%20'Finance%20Kasetsart%20University.'"
    "%20The%20Shiba%20Inu%20is%20facing%20to%20the%20right%2C"
    "%20wearing%20hea.webp"
)

display(Image(url=IMAGE_URL, width=1000))
print("Congratulation!!!! You did it 👍!!!")

#